# 04 — Correlation Analysis: Weather vs Price

**Objective**: Quantify the relationship between mesoregion-level weather anomalies and arabica coffee futures returns.

**Methods**: Cross-correlation function (CCF), Granger causality tests, OLS regression.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))

import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from src.data.merge_data import load_combined
from src.config import MESOREGIONS
from src.analysis.correlation import compute_ccf, run_correlation
from src.analysis.models import run_models
from src.visualization.correlation_plots import (
    lag_correlation_heatmap,
    granger_summary_table,
)

combined = load_combined()
print(f"Combined panel: {len(combined)} days × {len(combined.columns)} columns")
print(f"Date range: {combined.index.min()} to {combined.index.max()}")

## 4.1 Cross-Correlation: Temperature vs Returns

For each mesoregion, compute correlation between daily temperature anomaly and coffee returns at lags 0–180 days.

Positive lag = temperature leads price.

In [ ]:
run_correlation()

## 4.2 Lag-Correlation Heatmap

Visual summary: rows = top 6 mesoregions, columns = lag in days, color = correlation strength.

In [ ]:
lag_correlation_heatmap().show()

## 4.3 Granger Causality

Does a mesoregion's temperature anomaly statistically Granger-cause coffee returns at lag 5?

In [ ]:
granger_summary_table().show()

## 4.4 Detailed Mesoregion CCF Plot

Pick a mesoregion and see its full cross-correlation function.

In [ ]:
returns = combined["Close"].pct_change().dropna()

top_mesos = ["31_10", "31_05", "31_12", "35_02", "32_04", "29_06"]
fig = go.Figure()
colors = ["#4a3728", "#8b7355", "#c8b89a", "#e07a5f", "#d4a373", "#3d405b"]

for i, mk in enumerate(top_mesos):
    col = f"{mk}_temp_max_anom"
    if col not in combined.columns:
        continue
    ccf = compute_ccf(combined[col], returns, max_lag=90)
    fig.add_trace(go.Scatter(
        x=ccf["lag"], y=ccf["correlation"],
        mode="lines",
        name=MESOREGIONS.get(mk, {}).get("meso_name", mk),
        line=dict(color=colors[i % len(colors)], width=1.5),
    ))

fig.add_hline(y=0, line_dash="dash", line_color="grey", opacity=0.5)
fig.update_layout(
    title="Cross-Correlation: Temp Anomaly → Coffee Returns (0-90 day lag)",
    xaxis_title="Lag (days)",
    yaxis_title="Correlation",
    height=500,
    hovermode="x unified",
)
fig.show()

## 4.5 Regression Models

Run the 5 OLS specifications with Newey-West standard errors.

In [ ]:
run_models()

## Key Observations

- Fill in your observations here after running the cells above.